## Writing to silver — save modes, partitioning, idempotency

The last step of every transformation: land the result in a governed silver table. Two write patterns the exam tests.

**Full overwrite** — the silver build is *idempotent*: re-running rebuilds it from bronze, so a retry is safe.

```python
(silver.write
       .mode("overwrite")
       .option("overwriteSchema", "true")   # only when the schema changed
       .saveAsTable("fintech_dev.silver.card_transactions"))
```

Save modes: `append` adds rows, `overwrite` replaces the table, `error`/`errorifexists` (the default) fails if it exists, `ignore` no-ops if it exists.

**Upsert with `MERGE INTO`** — silver tracks updates from bronze CDC instead of a full rebuild:

```sql
MERGE INTO fintech_dev.silver.card_transactions AS t
USING bronze_updates                            AS s
   ON t.transaction_id = s.transaction_id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

**Layout decisions:**

- **New tables → Liquid Clustering** on the natural query keys: `.clusterBy("customer_id", "transaction_at")` (module 02). It replaces both `partitionBy` and `ZORDER`.
- **Legacy tables** may still use `partitionBy("transaction_date")` — but don't reach for it on new tables.
- **Enable Predictive Optimization** at the catalog level so silver stays compact without a manual `OPTIMIZE` job (module 08).

**Idempotency is the throughline:** a full-overwrite rebuild is safe to retry; a `MERGE` keyed on the natural key is safe to re-apply. Either way, re-running the job converges to the same silver.
